In [1]:
import torch
import ultralytics
from ultralytics import YOLO

print(f"Ultralytics version: {ultralytics.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")

Ultralytics version: 8.3.232
PyTorch version: 2.3.1
GPU: NVIDIA GeForce GTX 1650 SUPER


In [2]:
model = YOLO('yolo11s-seg.pt') 

results = model.train(
    data=r"D:\projeto_placentas_clayton\dev\projeto-placentas\v2\data_v2.yaml",
    epochs=120,
    imgsz=640,
    batch=2,             # VRAM 4GB
    patience=30,
    device=0,
    name='placentas_v11_aug_v2',
    
    # multitask: contar e medir
    retina_masks=True,   # medicao de bordas hi-res
    overlap_mask=False,  # contagem sem sobreposicao
    mask_ratio=1,        # mantem resolucao da mascara
    
    # augmentation
    degrees=90.0,        
    flipud=0.5,
    fliplr=0.5,
    mosaic=1.0,          
    mixup=0.1,           
    scale=0.5,           
    hsv_s=0.7,
)

New https://pypi.org/project/ultralytics/8.4.21 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.232  Python-3.10.19 torch-2.3.1 CUDA:0 (NVIDIA GeForce GTX 1650 SUPER, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\projeto_placentas_clayton\dev\projeto-placentas\v2\data_v2.yaml, degrees=90.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=120, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=1, max_det=300, mixup=0.1, mode=train, model=yolo11s-seg.pt, momentum=0.937, mosaic=1.0, multi_sca

In [ ]:
import numpy as np
from ultralytics import YOLO

model = YOLO(r'D:\projeto_placentas_clayton\dev\projeto-placentas\v2\runs\segment\placentas_v11_aug_v2\weights\best.pt')

stats = model.val(plots=True, verbose=False)

print("=== INSPECTING curves_results ===")
for i, result in enumerate(stats.seg.curves_results):
    x, y, xlabel, ylabel = result
    print(f"Index {i}: xlabel='{xlabel}', ylabel='{ylabel}', x.shape={np.array(x).shape}, y.shape={np.array(y).shape}")

print("\n=== INSPECTING box curves_results ===")
for i, result in enumerate(stats.box.curves_results):
    x, y, xlabel, ylabel = result
    print(f"Index {i}: xlabel='{xlabel}', ylabel='{ylabel}', x.shape={np.array(x).shape}, y.shape={np.array(y).shape}")

Ultralytics 8.3.232  Python-3.10.19 torch-2.3.1 CUDA:0 (NVIDIA GeForce GTX 1650 SUPER, 4096MiB)
YOLO11s-seg summary (fused): 113 layers, 10,067,203 parameters, 0 gradients
val: Fast image access  (ping: 0.10.0 ms, read: 354.465.8 MB/s, size: 45.9 KB)
val: Scanning D:\projeto_placentas_clayton\dataset_v2_ready_for_yolo\valid\labels.cache... 27 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 27/27  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.0s/it 4.1s<11.7s
WARNING ConfusionMatrix plot failure: module 'numpy' has no attribute 'iterable'
WARNING ConfusionMatrix plot failure: module 'numpy' has no attribute 'iterable'
                   all         27        852      0.907      0.866      0.946       0.84      0.913      0.871      0.939       0.71
Speed: 3.1ms preprocess, 25.8ms inference, 0.0ms loss, 5.6ms postprocess per image
Results saved to D:\projeto_placentas_

In [ ]:
import numpy as np
from ultralytics import YOLO

model = YOLO(r'D:\projeto_placentas_clayton\dev\projeto-placentas\v2\runs\segment\placentas_v11_aug_v2\weights\best.pt')

stats = model.val(plots=True, verbose=False)

conf_values, f1_per_class, _, _ = stats.seg.curves_results[1]
conf_values = np.array(conf_values)
f1_per_class = np.array(f1_per_class)

mean_f1 = f1_per_class.mean(0)

best_idx = np.argmax(mean_f1)
optimal_conf = conf_values[best_idx]
peak_f1 = mean_f1[best_idx]

print("\n" + "="*50)
print("SEGMENTATION METRICS")
print("="*50)
print(f"Peak F1 Score:      {peak_f1:.4f}")
print(f"Optimal Confidence: {optimal_conf:.4f}")
print(f"mAP50:              {stats.seg.map50:.4f}")
print(f"mAP50-95:           {stats.seg.map:.4f}")
print("="*50)

# Box metrics
box_conf, box_f1_cls, _, _ = stats.box.curves_results[1]
box_conf = np.array(box_conf)
box_f1_cls = np.array(box_f1_cls)
box_optimal_conf = box_conf[np.argmax(box_f1_cls.mean(0))]
print(f"Reference Box Conf: {box_optimal_conf:.4f}")
print(f"Box mAP50:          {stats.box.map50:.4f}")

Ultralytics 8.3.232  Python-3.10.19 torch-2.3.1 CUDA:0 (NVIDIA GeForce GTX 1650 SUPER, 4096MiB)
YOLO11s-seg summary (fused): 113 layers, 10,067,203 parameters, 0 gradients
val: Fast image access  (ping: 0.00.0 ms, read: 377.344.7 MB/s, size: 47.5 KB)
val: Scanning D:\projeto_placentas_clayton\dataset_v2_ready_for_yolo\valid\labels.cache... 27 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 27/27 29.6Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.2s/it 4.4s<12.5s
WARNING ConfusionMatrix plot failure: module 'numpy' has no attribute 'iterable'
WARNING ConfusionMatrix plot failure: module 'numpy' has no attribute 'iterable'
                   all         27        852      0.907      0.866      0.946       0.84      0.913      0.871      0.939       0.71
Speed: 4.9ms preprocess, 25.6ms inference, 0.0ms loss, 5.5ms postprocess per image
Results saved to D:\projeto_p

In [7]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

MODEL_PATH = r'D:\projeto_placentas_clayton\dev\projeto-placentas\v2\runs\segment\placentas_v11_aug_v2\weights\best.pt'
IMAGES_DIR = r"D:\projeto_placentas_clayton\dataset_v2_ready_for_yolo\valid\images"
LABELS_DIR = r"D:\projeto_placentas_clayton\dataset_v2_ready_for_yolo\valid\labels"

OUTPUT_DIR = r"D:\projeto_placentas_clayton\dev\projeto-placentas\v2\iou_imaging"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CONF = 0.4595
IMG_W, IMG_H = 640, 640

model = YOLO(MODEL_PATH)
results = model.predict(source=IMAGES_DIR, conf=CONF, retina_masks=True, verbose=False)

for r in results:
    img_name = os.path.basename(r.path)
    label_path = os.path.join(LABELS_DIR, os.path.splitext(img_name)[0] + '.txt')

    img = cv2.cvtColor(cv2.imread(r.path), cv2.COLOR_BGR2RGB)

    # ── Build GT mask (combined) ──────────────────────────────────────────────
    gt_combined = np.zeros((IMG_H, IMG_W), dtype=np.uint8)
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f:
                parts = list(map(float, line.strip().split()))
                if len(parts) > 1:
                    poly = np.array(parts[1:]).reshape(-1, 2)
                    poly[:, 0] *= IMG_W
                    poly[:, 1] *= IMG_H
                    cv2.fillPoly(gt_combined, [poly.astype(np.int32)], 1)

    # ── Build AI mask (combined) ──────────────────────────────────────────────
    ai_combined = np.zeros((IMG_H, IMG_W), dtype=np.uint8)
    if r.masks is not None:
        masks = r.masks.data.cpu().numpy()
        ai_combined = (masks > 0.5).any(axis=0).astype(np.uint8)

    # ── Compute overall IoU ───────────────────────────────────────────────────
    intersection = np.logical_and(gt_combined, ai_combined).sum()
    union = np.logical_or(gt_combined, ai_combined).sum()
    iou = intersection / union if union > 0 else 0.0

    # ── Overlay: green=GT only, red=AI only, yellow=overlap ──────────────────
    overlay = img.copy()
    overlay[gt_combined == 1] = (overlay[gt_combined == 1] * 0.5 + np.array([0, 255, 0]) * 0.5).astype(np.uint8)
    overlay[ai_combined == 1] = (overlay[ai_combined == 1] * 0.5 + np.array([255, 0, 0]) * 0.5).astype(np.uint8)
    overlap = np.logical_and(gt_combined, ai_combined)
    overlay[overlap] = (overlay[overlap] * 0.5 + np.array([255, 255, 0]) * 0.5).astype(np.uint8)

    # ── Plot ──────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle(f"{img_name}  |  IoU: {iou:.3f}", fontsize=13)

    axes[0].imshow(img);            axes[0].set_title("Original");      axes[0].axis("off")
    axes[1].imshow(overlay);        axes[1].set_title("GT=green  AI=red  Overlap=yellow"); axes[1].axis("off")

    diff = np.zeros((IMG_H, IMG_W, 3), dtype=np.uint8)
    diff[gt_combined == 1] = [0, 200, 0]    # GT only → green
    diff[ai_combined == 1] = [200, 0, 0]    # AI only → red
    diff[overlap]          = [255, 255, 0]  # overlap  → yellow
    axes[2].imshow(diff);           axes[2].set_title("Difference map"); axes[2].axis("off")

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"iou_viz_{os.path.splitext(img_name)[0]}.png"), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"{img_name:40s}  IoU: {iou:.4f}")

ROSILHA-M-B_003_jpg.rf.863a24cebe30a134937348de5cbcaa87.jpg  IoU: 0.8932
ROSILHA-M-B_009_jpg.rf.d2aa8881a0b4f54b4bdb89392b953e71.jpg  IoU: 0.8585
ROSILHA-M-D_006_jpg.rf.de71085683e9c7eb460be0f184fb31ac.jpg  IoU: 0.8616
ROSILHA-M-D_009_jpg.rf.b52035c8fe5fd87e1eed76dc4bbe6a14.jpg  IoU: 0.9227
ROSILHA-M-G_001_jpg.rf.3dab74a9f6c203f6aca8afc07a353bd2.jpg  IoU: 0.9081
ROSILHA-M-G_005_jpg.rf.afae2fb85329c0fe56679d51ca1c64fd.jpg  IoU: 0.8283
ROSILHA-M-G_017_jpg.rf.85fc7b8a7b63a2cbb64907a1c82d5d51.jpg  IoU: 0.7516
TORDILHA-B_009_jpg.rf.5902113a7929ec9592dc2b09471de461.jpg  IoU: 0.8783
TORDILHA-B_010_jpg.rf.698124083a3806b10e86f6304c2b5ce1.jpg  IoU: 0.9013
TORDILHA-B_012_jpg.rf.0b5caafb1952d960be9aa54f5506bf96.jpg  IoU: 0.8364
TORDILHA-D_001_jpg.rf.36e39c15aad81283f9bcc0f8afcd649f.jpg  IoU: 0.8244
TORDILHA-D_006_jpg.rf.d879d44ef1add0b327aa39f6650afc47.jpg  IoU: 0.6426
TORDILHA-G_009_jpg.rf.b279af9f6892d54790621c0f5458daf3.jpg  IoU: 0.8145
TOSTADA-B_003_jpg.rf.ef070508fe8cf028db79cd03961c37d8.jpg

In [8]:
import os
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO

MODEL_PATH = r'D:\projeto_placentas_clayton\dev\projeto-placentas\v2\runs\segment\placentas_v11_aug_v2\weights\best.pt'
IMAGES_DIR = r"D:\projeto_placentas_clayton\dataset_v2_ready_for_yolo\valid\images"
LABELS_DIR = r"D:\projeto_placentas_clayton\dataset_v2_ready_for_yolo\valid\labels"
OUTPUT_CSV = r"D:\projeto_placentas_clayton\dev\projeto-placentas\v2\placenta_instance_report.csv"
CONF = 0.4595
IMG_W, IMG_H = 640, 640
AREA_FACTOR = (50 / 72) ** 2
IOU_THRESHOLD = 0.5

model = YOLO(MODEL_PATH)
results = model.predict(source=IMAGES_DIR, conf=CONF, retina_masks=True, verbose=False)

all_rows = []

for r in results:
    img_name = os.path.basename(r.path)
    label_path = os.path.join(LABELS_DIR, os.path.splitext(img_name)[0] + '.txt')

    # ── GT instances ──────────────────────────────────────────────────────────
    gt_masks = []
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f:
                parts = list(map(float, line.strip().split()))
                if len(parts) > 1:
                    poly = np.array(parts[1:]).reshape(-1, 2)
                    poly[:, 0] *= IMG_W
                    poly[:, 1] *= IMG_H
                    m = np.zeros((IMG_H, IMG_W), dtype=np.uint8)
                    cv2.fillPoly(m, [poly.astype(np.int32)], 1)
                    gt_masks.append(m)

    # ── AI instances ──────────────────────────────────────────────────────────
    ai_masks = []
    if r.masks is not None:
        for mask in r.masks.data.cpu().numpy():
            ai_masks.append((mask > 0.5).astype(np.uint8))

    # ── Build full IoU matrix (AI x GT) ──────────────────────────────────────
    n_ai, n_gt = len(ai_masks), len(gt_masks)
    iou_matrix = np.zeros((n_ai, n_gt))

    for i, ai_mask in enumerate(ai_masks):
        for j, gt_mask in enumerate(gt_masks):
            intersection = np.logical_and(ai_mask, gt_mask).sum()
            union = np.logical_or(ai_mask, gt_mask).sum()
            iou_matrix[i, j] = intersection / union if union > 0 else 0.0

    # ── Optimal matching: highest IoU pairs first ─────────────────────────────
    matched_ai, matched_gt = set(), set()
    pairs = []

    if n_ai > 0 and n_gt > 0:
        # Sort all (ai, gt) pairs by IoU descending
        indices = np.dstack(np.unravel_index(np.argsort(-iou_matrix, axis=None), iou_matrix.shape))[0]
        for ai_idx, gt_idx in indices:
            if ai_idx in matched_ai or gt_idx in matched_gt:
                continue
            if iou_matrix[ai_idx, gt_idx] < IOU_THRESHOLD:
                break  # remaining pairs are all below threshold
            pairs.append((ai_idx, gt_idx, iou_matrix[ai_idx, gt_idx]))
            matched_ai.add(ai_idx)
            matched_gt.add(gt_idx)

    # ── True Positives ────────────────────────────────────────────────────────
    for ai_idx, gt_idx, iou in pairs:
        ai_area = int(ai_masks[ai_idx].sum())
        gt_area = int(gt_masks[gt_idx].sum())
        area_err_pct = (ai_area - gt_area) / gt_area * 100 if gt_area > 0 else 0
        all_rows.append({
            "Image":       img_name,
            "Type":        "TP",
            "IoU":         round(iou, 4),
            "GT_Area_um2": round(gt_area * AREA_FACTOR, 2),
            "AI_Area_um2": round(ai_area * AREA_FACTOR, 2),
            "Area_Err_%":  round(area_err_pct, 2),
        })

    # ── False Positives ───────────────────────────────────────────────────────
    for ai_idx, ai_mask in enumerate(ai_masks):
        if ai_idx not in matched_ai:
            all_rows.append({
                "Image":       img_name,
                "Type":        "FP",
                "IoU":         round(float(iou_matrix[ai_idx].max()) if n_gt > 0 else 0.0, 4),
                "GT_Area_um2": None,
                "AI_Area_um2": round(int(ai_mask.sum()) * AREA_FACTOR, 2),
                "Area_Err_%":  None,
            })

    # ── False Negatives ───────────────────────────────────────────────────────
    for gt_idx, gt_mask in enumerate(gt_masks):
        if gt_idx not in matched_gt:
            all_rows.append({
                "Image":       img_name,
                "Type":        "FN",
                "IoU":         0.0,
                "GT_Area_um2": round(int(gt_mask.sum()) * AREA_FACTOR, 2),
                "AI_Area_um2": None,
                "Area_Err_%":  None,
            })

# ── Report ────────────────────────────────────────────────────────────────────
df = pd.DataFrame(all_rows)
df.to_csv(OUTPUT_CSV, index=False)

tp = df[df.Type == "TP"]
fp = df[df.Type == "FP"]
fn = df[df.Type == "FN"]

precision = len(tp) / (len(tp) + len(fp)) if (len(tp) + len(fp)) > 0 else 0
recall    = len(tp) / (len(tp) + len(fn)) if (len(tp) + len(fn)) > 0 else 0

print("\n" + "="*55)
print("PER-INSTANCE REPORT")
print("="*55)
print(f"True Positives  (matched):      {len(tp)}")
print(f"False Positives (extra preds):  {len(fp)}")
print(f"False Negatives (missed GT):    {len(fn)}")
print(f"Precision:                      {precision:.4f}")
print(f"Recall:                         {recall:.4f}")
print("-"*55)
print(f"Mean IoU        (TP only):      {tp['IoU'].mean():.4f}")
print(f"Mean GT Area    (TP only):      {tp['GT_Area_um2'].mean():.2f} µm²")
print(f"Mean AI Area    (TP only):      {tp['AI_Area_um2'].mean():.2f} µm²")
print(f"Mean Area Error (TP only):      {tp['Area_Err_%'].mean():.2f}%")
print(f"Mean Abs Error  (TP only):      {tp['Area_Err_%'].abs().mean():.2f}%")
print("="*55)
print(f"Report saved to: {OUTPUT_CSV}")


PER-INSTANCE REPORT
True Positives  (matched):      743
False Positives (extra preds):  77
False Negatives (missed GT):    109
Precision:                      0.9061
Recall:                         0.8721
-------------------------------------------------------
Mean IoU        (TP only):      0.8809
Mean GT Area    (TP only):      2377.68 µm²
Mean AI Area    (TP only):      2389.24 µm²
Mean Area Error (TP only):      0.96%
Mean Abs Error  (TP only):      5.96%
Report saved to: D:\projeto_placentas_clayton\dev\projeto-placentas\v2\placenta_instance_report.csv


In [9]:
for r in results:
    if r.masks is not None:
        print(f"{os.path.basename(r.path)}: mask shape = {r.masks.data.shape}")
        break

ROSILHA-M-B_003_jpg.rf.863a24cebe30a134937348de5cbcaa87.jpg: mask shape = torch.Size([20, 640, 640])


In [10]:
# Pick a GT polygon from a label file and compute area two ways
# 1) via your pipeline
# 2) via the shoelace formula (exact analytical area of the polygon)

def shoelace_area(poly_normalized, img_w, img_h):
    """Exact polygon area in pixels using shoelace formula."""
    pts = poly_normalized.copy()
    pts[:, 0] *= img_w
    pts[:, 1] *= img_h
    x, y = pts[:, 0], pts[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))

# Test on first polygon of first label
label_path = r"D:\projeto_placentas_clayton\dataset_v2_ready_for_yolo\valid\labels\ROSILHA-M-B_003_jpg.rf.863a24cebe30a134937348de5cbcaa87.txt"
with open(label_path, 'r') as f:
    line = f.readline()

parts = list(map(float, line.strip().split()))
poly = np.array(parts[1:]).reshape(-1, 2)

# Method 1: rasterized (your pipeline)
m = np.zeros((640, 640), dtype=np.uint8)
cv2.fillPoly(m, [( poly * 640).astype(np.int32)], 1)
raster_area_um2 = m.sum() * (50/72)**2

# Method 2: shoelace (analytical)
shoelace_area_um2 = shoelace_area(poly, 640, 640) * (50/72)**2

print(f"Rasterized area: {raster_area_um2:.2f} µm²")
print(f"Shoelace area:   {shoelace_area_um2:.2f} µm²")
print(f"Difference:      {abs(raster_area_um2 - shoelace_area_um2) / shoelace_area_um2 * 100:.2f}%")

Rasterized area: 6305.94 µm²
Shoelace area:   6178.23 µm²
Difference:      2.07%
